In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np

# --- 1. Data Preparation (PyTorch Equivalent) ---

data = """ Jack and Jill went up the hill .\n To fetch a pail of water .\n Jack fell down and broke his crown .\n And Jill came tumbling after ."""
print("Data:", data, type(data))

data_splitted = data.split('\n') # returns a list of strings
print("Data_Splitted:", data_splitted, type(data_splitted))

# Custom Tokenizer to replicate Keras Tokenizer behavior
class PyTorchTokenizer:
    def __init__(self, filters='!"#$%&()*+,-/:;<=>?@[\]^_`{|}~'):
        self.word_index = {}
        self.index_word = {0: 'PAD'}
        self.filters = filters
        self.word_count = {}

    def fit_on_texts(self, texts):
        current_index = 1
        for text in texts:
            # Basic cleaning, remove filters, convert to lowercase
            cleaned_text = text.lower()
            for char in self.filters:
                cleaned_text = cleaned_text.replace(char, '')
            words = cleaned_text.split()
            for word in words:
                if word not in self.word_index:
                    self.word_index[word] = current_index
                    self.index_word[current_index] = word
                    current_index += 1
                self.word_count[word] = self.word_count.get(word, 0) + 1

    def texts_to_sequences(self, texts):
        sequences = []
        for text in texts:
            cleaned_text = text.lower()
            for char in self.filters:
                cleaned_text = cleaned_text.replace(char, '')
            words = cleaned_text.split()
            seq = [self.word_index[word] for word in words if word in self.word_index]
            sequences.append(seq)
        return sequences

# Instantiate and fit tokenizer
tokenizer_pt = PyTorchTokenizer()
tokenizer_pt.fit_on_texts(data_splitted)
print("Word Indices (PyTorch):", tokenizer_pt.word_index)

vocab_size_pt = len(tokenizer_pt.word_index) + 1
print("Vocab Size (PyTorch):", vocab_size_pt)

sequences_pt = tokenizer_pt.texts_to_sequences(data_splitted)
print("Sequences (PyTorch):", sequences_pt, type(sequences_pt), len(sequences_pt))

X_pt = []
y_pt = []

for seq in sequences_pt:
    X_pt.append(seq[:-1])
    y_pt.append(seq)

print("X_pt=", X_pt, "y_pt=", y_pt, type(X_pt), type(y_pt))

# Padding sequences
maxlen_pt = max([len(sequence) for sequence in X_pt])
print("Maxlen (PyTorch):", maxlen_pt)

# Keras pad_sequences defaults to 'pre' padding with 0. Maxlen is +1 for input X
def pad_sequences_pt(sequences, maxlen, padding='pre', value=0):
    padded_sequences = []
    for seq in sequences:
        if len(seq) >= maxlen:
            padded_sequences.append(seq[-maxlen:]) # Truncate if too long (Keras default is 'pre')
        else:
            pad_length = maxlen - len(seq)
            if padding == 'pre':
                padded_sequences.append([value] * pad_length + seq)
            elif padding == 'post':
                padded_sequences.append(seq + [value] * pad_length)
    return np.array(padded_sequences)

X_padded_pt = pad_sequences_pt(X_pt, maxlen=maxlen_pt + 1, padding='pre') # +1 to have 0 as the first input
y_padded_pt = pad_sequences_pt(y_pt, maxlen=maxlen_pt + 1, padding='pre')

print("X_padded_pt:", X_padded_pt, type(X_padded_pt), X_padded_pt.shape)
print("y_padded_pt:", y_padded_pt, type(y_padded_pt), y_padded_pt.shape)

# Convert to PyTorch tensors
X_tensor = torch.tensor(X_padded_pt, dtype=torch.long)

# y needs to be one-hot encoded for categorical_crossentropy, shape (batch, seq_len, vocab_size)
y_one_hot = F.one_hot(torch.tensor(y_padded_pt, dtype=torch.long), num_classes=vocab_size_pt).float()

print("X_tensor shape:", X_tensor.shape)
print("y_one_hot shape:", y_one_hot.shape)


# --- 2. Model Definition (PyTorch Equivalent) ---

class PyTorchRNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size):
        super(PyTorchRNN, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.rnn = nn.RNN(embedding_dim, hidden_size, batch_first=True) # batch_first=True for (batch, seq, feature)
        self.dense = nn.Linear(hidden_size, vocab_size)

    def forward(self, x):
        embedded = self.embedding(x)
        output, _ = self.rnn(embedded) # _ is the hidden state, not used here for simple RNN
        # Apply dense layer to each time step's output
        output = self.dense(output)
        return output


embedding_dim = 10
hidden_size = 50
model_pt = PyTorchRNN(vocab_size_pt, embedding_dim, hidden_size)
print(model_pt)


# --- 3. Training Loop (PyTorch Equivalent) ---

optimizer_pt = optim.RMSprop(model_pt.parameters())
# CrossEntropyLoss expects (N, C, ...) for input and (N, ...) for target
# Our output is (batch, seq_len, vocab_size), target is (batch, seq_len, vocab_size) one-hot
# We need to reshape and then apply F.log_softmax and NLLLoss, or flatten and use CrossEntropyLoss

# For simplicity, we will flatten and use CrossEntropyLoss which combines LogSoftmax and NLLLoss
# Input to CrossEntropyLoss should be (N, C) and target (N) where N is total number of elements

def train_pytorch_model(model, X_train, y_train_one_hot, epochs, optimizer):
    model.train() # Set the model to training mode
    criterion = nn.CrossEntropyLoss() # This implicitly applies LogSoftmax and then NLLLoss

    # Reshape y_train_one_hot to (batch_size * seq_len, vocab_size)
    # And get the target indices from the one-hot encoding for CrossEntropyLoss
    y_target_indices = torch.argmax(y_train_one_hot, dim=-1) # (batch_size, seq_len)

    for epoch in range(epochs):
        optimizer.zero_grad()

        # Forward pass
        outputs = model(X_train) # (batch_size, seq_len, vocab_size)

        # Reshape outputs for CrossEntropyLoss: (batch_size * seq_len, vocab_size)
        outputs_flat = outputs.view(-1, vocab_size_pt)
        # Reshape target indices: (batch_size * seq_len)
        targets_flat = y_target_indices.view(-1)

        loss = criterion(outputs_flat, targets_flat)
        loss.backward() # Backward pass
        optimizer.step() # Optimize

        if (epoch + 1) % 50 == 0:
            print(f'Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}')

epochs_pt = 200
train_pytorch_model(model_pt, X_tensor, y_one_hot, epochs_pt, optimizer_pt)

print("\n--- PyTorch Model Training Complete ---")


# --- 4. Probability Calculation Function (PyTorch Equivalent) ---

def prob_of_input_sentence_pt(model, tokenizer, sentence, vocab_size):
    model.eval() # Set the model to evaluation mode
    with torch.no_grad(): # Disable gradient calculations
        print("Input Sentence:", sentence)

        # Tokenize the sentence using the custom tokenizer
        encoded_list = tokenizer.texts_to_sequences([sentence])[0]

        # Keras `pad_sequences` behavior: inserts 0 at the beginning if input is shorter than maxlen.
        # Our X was padded with +1, so we need to match that.
        # The original Keras code inserted a 0 at the beginning manually, then padded.
        # We need to consider how the original maxlen was defined (maxlen_pt is for X_pt, which is already `seq[:-1]`).
        # The `X_tensor` was padded to `maxlen_pt + 1`.
        # So, the input to the model should be of `maxlen_pt + 1`.

        # First, ensure the '0' prefix for empty/short sequences as in Keras example
        # Original Keras code: encoded.insert(0,0) - this inserts 0 irrespective of actual length, then pad_sequences
        # So if `sentence` tokenizes to `[1, 2, 3]`, encoded becomes `[0, 1, 2, 3]` and then padded.

        input_seq_for_prob = [0] + encoded_list # Prepend 0 as in original Keras

        # Pad to the length the model expects, which is maxlen_pt + 1
        # Use our custom pad_sequences_pt with 'pre' padding
        padded_input = pad_sequences_pt([input_seq_for_prob], maxlen=maxlen_pt + 1, padding='pre')[0]

        encoded_tensor = torch.tensor(padded_input, dtype=torch.long).unsqueeze(0) # Add batch dimension
        print("Encoded (PyTorch):", encoded_tensor, encoded_tensor.shape)

        # Get model predictions (logits)
        logits = model(encoded_tensor) # (1, seq_len, vocab_size)

        # Apply softmax to get probabilities for each token at each step
        prob = F.softmax(logits, dim=-1) # (1, seq_len, vocab_size)
        print("Prob (PyTorch):", prob, prob.shape)

        probability = 1.0

        # Iterate through the sequence to calculate the joint probability
        # The Keras code calculated P(word_i | word_0...word_{i-1})
        # and multiplied these probabilities. This corresponds to the probability
        # of the actual next word in the sequence, given the previous ones.

        # The original Keras code iterates from i=0 to prob.shape[1]-1 (length-1)
        # and uses encoded[0, i+1] as the target word.

        # So for an input sequence `s_0, s_1, ..., s_N`, the model predicts `p_0, p_1, ..., p_N`
        # `p_0` is prob of `s_1` given `s_0`
        # `p_1` is prob of `s_2` given `s_0, s_1` and so on.
        # The Keras `encoded[0, i+1]` is the actual next word at position `i+1`.

        for i in range(prob.shape[1] - 1):
            # prob[0, i, :] gives the probability distribution for the next word at step i
            # encoded_tensor[0, i+1] is the actual next word index in the input sequence

            # Ensure the index is within bounds of vocab_size
            target_word_index = int(encoded_tensor[0, i+1].item())
            if target_word_index >= vocab_size:
                # This case might happen if token not in vocab, or it's a padding zero.
                # If it's a padding zero, its probability should be 1 if it's the target
                # Or we skip it if it's not a real word. Keras handles padding '0' gracefully.
                # We will assume 0 (PAD) has a probability of 1 if it's the target (for padding effect)
                # But if a word from the sentence itself is not in vocab, its prob is 0.
                # For this specific Keras example, all words are in vocab.
                # If target_word_index is 0 (PAD), we treat it as 1 for simplicity of padding effect.
                if target_word_index == 0:
                    word_prob = 1.0 # Or maybe skip. Original Keras would try to predict prob of 0.
                                    # For padding, it usually means we're done with the sentence.
                                    # The original Keras prob was calculated for 0 as well.
                else:
                    word_prob = 0.0 # Word not in vocab
            else:
                word_prob = prob[0, i, target_word_index].item()
            probability *= word_prob

        print("Probability of Sentence ", "\"" + sentence + "\"", "is:", probability)


print("-------------------Probability of Input Sentence (PyTorch)------------------------------")
prob_of_input_sentence_pt(model_pt, tokenizer_pt, "Jack and Jill Went", vocab_size_pt)
prob_of_input_sentence_pt(model_pt, tokenizer_pt, "and Jill Went up", vocab_size_pt)
prob_of_input_sentence_pt(model_pt, tokenizer_pt, "went up the hill", vocab_size_pt)
prob_of_input_sentence_pt(model_pt, tokenizer_pt, "jack and Jill Went up the hill .", vocab_size_pt)
prob_of_input_sentence_pt(model_pt, tokenizer_pt, "to fetch a pail", vocab_size_pt)
prob_of_input_sentence_pt(model_pt, tokenizer_pt, "fetch a pail", vocab_size_pt)
prob_of_input_sentence_pt(model_pt, tokenizer_pt, "to fetch a pail of water", vocab_size_pt)
prob_of_input_sentence_pt(model_pt, tokenizer_pt, "jack fell down and", vocab_size_pt)
prob_of_input_sentence_pt(model_pt, tokenizer_pt, "jack fell down and broke", vocab_size_pt)
prob_of_input_sentence_pt(model_pt, tokenizer_pt, "fell down and broke", vocab_size_pt)
prob_of_input_sentence_pt(model_pt, tokenizer_pt, "jack fell down and broke his crown", vocab_size_pt)
prob_of_input_sentence_pt(model_pt, tokenizer_pt, "and jill came tumbling", vocab_size_pt)
prob_of_input_sentence_pt(model_pt, tokenizer_pt, "g after", vocab_size_pt)
prob_of_input_sentence_pt(model_pt, tokenizer_pt, "and jill came tumbling after .", vocab_size_pt)
prob_of_input_sentence_pt(model_pt, tokenizer_pt, "and jill came walkings after .", vocab_size_pt)
print("-------------------------------------------------------------------------------")

<>:17: SyntaxWarning: invalid escape sequence '\]'
<>:17: SyntaxWarning: invalid escape sequence '\]'
/tmp/ipykernel_10858/1083552682.py:17: SyntaxWarning: invalid escape sequence '\]'
  def __init__(self, filters='!"#$%&()*+,-/:;<=>?@[\]^_`{|}~'):


Data:  Jack and Jill went up the hill .
 To fetch a pail of water .
 Jack fell down and broke his crown .
 And Jill came tumbling after . <class 'str'>
Data_Splitted: [' Jack and Jill went up the hill .', ' To fetch a pail of water .', ' Jack fell down and broke his crown .', ' And Jill came tumbling after .'] <class 'list'>
Word Indices (PyTorch): {'jack': 1, 'and': 2, 'jill': 3, 'went': 4, 'up': 5, 'the': 6, 'hill': 7, '.': 8, 'to': 9, 'fetch': 10, 'a': 11, 'pail': 12, 'of': 13, 'water': 14, 'fell': 15, 'down': 16, 'broke': 17, 'his': 18, 'crown': 19, 'came': 20, 'tumbling': 21, 'after': 22}
Vocab Size (PyTorch): 23
Sequences (PyTorch): [[1, 2, 3, 4, 5, 6, 7, 8], [9, 10, 11, 12, 13, 14, 8], [1, 15, 16, 2, 17, 18, 19, 8], [2, 3, 20, 21, 22, 8]] <class 'list'> 4
X_pt= [[1, 2, 3, 4, 5, 6, 7], [9, 10, 11, 12, 13, 14], [1, 15, 16, 2, 17, 18, 19], [2, 3, 20, 21, 22]] y_pt= [[1, 2, 3, 4, 5, 6, 7, 8], [9, 10, 11, 12, 13, 14, 8], [1, 15, 16, 2, 17, 18, 19, 8], [2, 3, 20, 21, 22, 8]] <class 'l

In [5]:
#Some Changes done
# 1. Add Unk in word_index dictionary
# 2. If any new words come then it is assigned 'UNK'

In [6]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np


data = """ Jack and Jill went up the hill .\n To fetch a pail of water .\n Jack fell down and broke his crown .\n And Jill came tumbling after ."""
print("Data:", data, type(data))

data_splitted = data.split('\n')
print("Data_Splitted:", data_splitted, type(data_splitted))

class PyTorchTokenizer:
    def __init__(self, filters='!"#$%&()*+,-/:;<=>?@[\]^_`{|}~'):
        self.word_index = {"UNK": 1}
        self.index_word = {0: 'PAD',1: 'UNK'}
        self.filters = filters
        self.word_count = {}

    def fit_on_texts(self, texts):
        current_index = 2

        for text in texts:
            cleaned_text = text.lower()
            for char in self.filters:
                cleaned_text = cleaned_text.replace(char, '')
            words = cleaned_text.split()
            for word in words:
                if word not in self.word_index:
                    self.word_index[word] = current_index
                    self.index_word[current_index] = word
                    current_index += 1
                self.word_count[word] = self.word_count.get(word, 0) + 1


    def texts_to_sequences(self, texts):
        sequences = []
        for text in texts:
            cleaned_text = text.lower()
            for char in self.filters:
                cleaned_text = cleaned_text.replace(char, '')
            words = cleaned_text.split()

            seq = []

            for word in words:
                if word in self.word_index:
                  seq.append(self.word_index[word])
                else:
                  seq.append(self.word_index["UNK"])

            sequences.append(seq)
        return sequences



tokenizer_pt = PyTorchTokenizer()
tokenizer_pt.fit_on_texts(data_splitted)
print("Word Indices (PyTorch):", tokenizer_pt.word_index)

vocab_size_pt = len(tokenizer_pt.word_index) + 1
print("Vocab Size (PyTorch):", vocab_size_pt)

sequences_pt = tokenizer_pt.texts_to_sequences(data_splitted)
print("Sequences (PyTorch):", sequences_pt, type(sequences_pt), len(sequences_pt))

X_pt = []
y_pt = []

for seq in sequences_pt:
    X_pt.append(seq[:-1])
    y_pt.append(seq)

print("X_pt=", X_pt, "y_pt=", y_pt, type(X_pt), type(y_pt))


maxlen_pt = max([len(sequence) for sequence in X_pt])
print("Maxlen (PyTorch):", maxlen_pt)

def pad_sequences_pt(sequences, maxlen, padding='pre', value=0):
    padded_sequences = []
    for seq in sequences:
        if len(seq) >= maxlen:
            padded_sequences.append(seq[-maxlen:])
        else:
            pad_length = maxlen - len(seq)
            if padding == 'pre':
                padded_sequences.append([value] * pad_length + seq)
            elif padding == 'post':
                padded_sequences.append(seq + [value] * pad_length)
    return np.array(padded_sequences)

X_padded_pt = pad_sequences_pt(X_pt, maxlen=maxlen_pt + 1, padding='pre') # +1 to have 0 as the first input
y_padded_pt = pad_sequences_pt(y_pt, maxlen=maxlen_pt + 1, padding='pre')

print("X_padded_pt:", X_padded_pt, type(X_padded_pt), X_padded_pt.shape)
print("y_padded_pt:", y_padded_pt, type(y_padded_pt), y_padded_pt.shape)

X_tensor = torch.tensor(X_padded_pt, dtype=torch.long)

y_one_hot = F.one_hot(torch.tensor(y_padded_pt, dtype=torch.long), num_classes=vocab_size_pt).float()

print("X_tensor shape:", X_tensor.shape)
print("y_one_hot shape:", y_one_hot.shape)
print("y_one_hot:", y_one_hot)



class PyTorchRNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size):
        super(PyTorchRNN, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.rnn = nn.RNN(embedding_dim, hidden_size, batch_first=True) # batch_first=True for (batch, seq, feature)
        self.dense = nn.Linear(hidden_size, vocab_size)

    def forward(self, x):
        embedded = self.embedding(x)
        output, _ = self.rnn(embedded)
        output = self.dense(output)
        return output


embedding_dim = 10
hidden_size = 50
model_pt = PyTorchRNN(vocab_size_pt, embedding_dim, hidden_size)
print(model_pt)

optimizer_pt = optim.RMSprop(model_pt.parameters())

def train_pytorch_model(model, X_train, y_train_one_hot, epochs, optimizer):
    model.train()
    criterion = nn.CrossEntropyLoss()

    y_target_indices = torch.argmax(y_train_one_hot, dim=-1)

    for epoch in range(epochs):
        optimizer.zero_grad()

        # Forward pass
        outputs = model(X_train) # (batch_size, seq_len, vocab_size)

        # Reshape outputs for CrossEntropyLoss: (batch_size * seq_len, vocab_size)
        outputs_flat = outputs.view(-1, vocab_size_pt)
        # Reshape target indices: (batch_size * seq_len)
        targets_flat = y_target_indices.view(-1)

        loss = criterion(outputs_flat, targets_flat)
        loss.backward()
        optimizer.step() # Optimize

        if (epoch + 1) % 50 == 0:
            print(f'Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}')

epochs_pt = 200
train_pytorch_model(model_pt, X_tensor, y_one_hot, epochs_pt, optimizer_pt)

print("\n--- PyTorch Model Training Complete ---")



def prob_of_input_sentence_pt(model, tokenizer, sentence, vocab_size):
    model.eval() # Set the model to evaluation mode
    with torch.no_grad(): # Disable gradient calculations
        print("Input Sentence:", sentence)

        encoded_list = tokenizer.texts_to_sequences([sentence])[0]

        input_seq_for_prob = [0] + encoded_list # Prepend 0 as in original Keras
        print("input_seq_for_prob: ", input_seq_for_prob)

        padded_input = pad_sequences_pt([input_seq_for_prob], maxlen=maxlen_pt + 1, padding='pre')[0]

        encoded_tensor = torch.tensor(padded_input, dtype=torch.long).unsqueeze(0) # Add batch dimension
        print("Encoded (PyTorch):", encoded_tensor, encoded_tensor.shape)

        logits = model(encoded_tensor) # (1, seq_len, vocab_size)
        print("logits: ", logits)

        prob = F.softmax(logits, dim=-1) # (1, seq_len, vocab_size)
        print("Prob (PyTorch):", prob, prob.shape)

        probability = 1.0


        for i in range(prob.shape[1] - 1):

            target_word_index = int(encoded_tensor[0, i+1].item())
            if target_word_index >= vocab_size:

                if target_word_index == 0:
                    word_prob = 1.0
                else:
                    word_prob = 0.0 # Word not in vocab
            else:
                word_prob = prob[0, i, target_word_index].item()
            probability *= word_prob

        print("Probability of Sentence ", "\"" + sentence + "\"", "is:", probability)


print("-------------------Probability of Input Sentence (PyTorch)------------------------------")
prob_of_input_sentence_pt(model_pt, tokenizer_pt, "Jack and Jill Went", vocab_size_pt)
prob_of_input_sentence_pt(model_pt, tokenizer_pt, "and Jill Went up", vocab_size_pt)
prob_of_input_sentence_pt(model_pt, tokenizer_pt, "went up the hill", vocab_size_pt)
prob_of_input_sentence_pt(model_pt, tokenizer_pt, "jack and Jill Went up the hill .", vocab_size_pt)
prob_of_input_sentence_pt(model_pt, tokenizer_pt, "to fetch a pail", vocab_size_pt)
prob_of_input_sentence_pt(model_pt, tokenizer_pt, "fetch a pail", vocab_size_pt)
prob_of_input_sentence_pt(model_pt, tokenizer_pt, "to fetch a pail of water", vocab_size_pt)
prob_of_input_sentence_pt(model_pt, tokenizer_pt, "jack fell down and", vocab_size_pt)
prob_of_input_sentence_pt(model_pt, tokenizer_pt, "jack fell down and broke", vocab_size_pt)
prob_of_input_sentence_pt(model_pt, tokenizer_pt, "fell down and broke", vocab_size_pt)
prob_of_input_sentence_pt(model_pt, tokenizer_pt, "jack fell down and broke his crown", vocab_size_pt)
prob_of_input_sentence_pt(model_pt, tokenizer_pt, "and jill came tumbling", vocab_size_pt)
prob_of_input_sentence_pt(model_pt, tokenizer_pt, "g after", vocab_size_pt)
prob_of_input_sentence_pt(model_pt, tokenizer_pt, "and jill came tumbling after .", vocab_size_pt)
prob_of_input_sentence_pt(model_pt, tokenizer_pt, "and jill came walkings after .", vocab_size_pt)
print("-------------------------------------------------------------------------------")

<>:16: SyntaxWarning: invalid escape sequence '\]'
<>:16: SyntaxWarning: invalid escape sequence '\]'
/tmp/ipykernel_10858/2160416410.py:16: SyntaxWarning: invalid escape sequence '\]'
  def __init__(self, filters='!"#$%&()*+,-/:;<=>?@[\]^_`{|}~'):


Data:  Jack and Jill went up the hill .
 To fetch a pail of water .
 Jack fell down and broke his crown .
 And Jill came tumbling after . <class 'str'>
Data_Splitted: [' Jack and Jill went up the hill .', ' To fetch a pail of water .', ' Jack fell down and broke his crown .', ' And Jill came tumbling after .'] <class 'list'>
Word Indices (PyTorch): {'UNK': 1, 'jack': 2, 'and': 3, 'jill': 4, 'went': 5, 'up': 6, 'the': 7, 'hill': 8, '.': 9, 'to': 10, 'fetch': 11, 'a': 12, 'pail': 13, 'of': 14, 'water': 15, 'fell': 16, 'down': 17, 'broke': 18, 'his': 19, 'crown': 20, 'came': 21, 'tumbling': 22, 'after': 23}
Vocab Size (PyTorch): 24
Sequences (PyTorch): [[2, 3, 4, 5, 6, 7, 8, 9], [10, 11, 12, 13, 14, 15, 9], [2, 16, 17, 3, 18, 19, 20, 9], [3, 4, 21, 22, 23, 9]] <class 'list'> 4
X_pt= [[2, 3, 4, 5, 6, 7, 8], [10, 11, 12, 13, 14, 15], [2, 16, 17, 3, 18, 19, 20], [3, 4, 21, 22, 23]] y_pt= [[2, 3, 4, 5, 6, 7, 8, 9], [10, 11, 12, 13, 14, 15, 9], [2, 16, 17, 3, 18, 19, 20, 9], [3, 4, 21, 22, 23,

In [ ]:
#  Explaination with Example
"""
 if len(seq) >= maxlen:
            padded_sequences.append(seq[-maxlen:])


seq = [1, 2, 3, 4, 5, 6, 7]
maxlen = 5
len(seq) = 7 ≥ 5  → True
seq[-5:] = [3, 4, 5, 6, 7]

padded_sequences.append([3, 4, 5, 6, 7])

"""

In [ ]:
"""
Input Sentence:
"Jack and Jill went up the hill"

        ↓ Tokenization
[2, 3, 4, 5, 6, 7, 8]

        ↓ Create X and y
X = [2, 3, 4, 5, 6, 7]
y = [2, 3, 4, 5, 6, 7, 8]

        ↓ Padding (+1)
X = [0, 2, 3, 4, 5, 6, 7]
y = [2, 3, 4, 5, 6, 7, 8]

        ↓ Embedding Layer
(1,7) → (1,7,10)

        ↓ RNN Layer
(1,7,10) → (1,7,50)

        ↓ Dense Layer
(1,7,50) → (1,7,vocab_size)

        ↓ Softmax
Probabilities for each word

        ↓ Output
Predict next word at each step


Sentence probability is calculated by multiplying probabilities of each word given previous words.

"""


"""
Objective of this practical:
This is a language model built using RNN. It converts text into sequences, uses embedding to represent words, and RNN to
learn sequence patterns. The model predicts the next word at each step, and we calculate sentence probability
using chain rule of probability.

"""

In [ ]:
"""
self.rnn = nn.RNN(input_size, hidden_size, batch_first=True)

What does batch_first=True mean?
It tells PyTorch how your input tensor is arranged

With batch_first=True
(batch, seq_len, features)

Input shape:
(2, 3, 4)

Meaning:
2 sentences
each has 3 words
each word has 4 features

[
  [word1, word2, word3],   ← sentence 1
  [word1, word2, word3]    ← sentence 2
]

With batch_first=False (default)
(seq_len, batch, features)

Input shape:
(3, 2, 4)

Meaning:
3 time steps
each step has 2 sentences
each word has 4 features

[
  [word1_s1, word1_s2],
  [word2_s1, word2_s2],
  [word3_s1, word3_s2]
]


In [ ]:
# Build Language Model using RNN, LSTM, GRU

In [7]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np


data = """ Jack and Jill went up the hill .\n To fetch a pail of water .\n Jack fell down and broke his crown .\n And Jill came tumbling after ."""

data_splitted = data.split('\n')


class PyTorchTokenizer:
    def __init__(self, filters='!"#$%&()*+,-/:;<=>?@[\]^_`{|}~'):
        self.word_index = {"UNK": 1}
        self.index_word = {0: 'PAD',1: 'UNK'}
        self.filters = filters

    def fit_on_texts(self, texts):
        current_index = 2
        for text in texts:
            cleaned_text = text.lower()
            for char in self.filters:
                cleaned_text = cleaned_text.replace(char, '')
            words = cleaned_text.split()
            for word in words:
                if word not in self.word_index:
                    self.word_index[word] = current_index
                    self.index_word[current_index] = word
                    current_index += 1

    def texts_to_sequences(self, texts):
        sequences = []
        for text in texts:
            cleaned_text = text.lower()
            for char in self.filters:
                cleaned_text = cleaned_text.replace(char, '')
            words = cleaned_text.split()

            seq = []
            for word in words:
                seq.append(self.word_index.get(word, 1))
            sequences.append(seq)
        return sequences


tokenizer_pt = PyTorchTokenizer()
tokenizer_pt.fit_on_texts(data_splitted)

vocab_size_pt = len(tokenizer_pt.word_index) + 1

sequences_pt = tokenizer_pt.texts_to_sequences(data_splitted)



X_pt = []
y_pt = []

for seq in sequences_pt:
    X_pt.append(seq[:-1])
    y_pt.append(seq)



maxlen_pt = max([len(sequence) for sequence in X_pt])

def pad_sequences_pt(sequences, maxlen, padding='pre', value=0):
    padded_sequences = []
    for seq in sequences:
        if len(seq) >= maxlen:
            padded_sequences.append(seq[-maxlen:])
        else:
            pad_length = maxlen - len(seq)
            if padding == 'pre':
                padded_sequences.append([value] * pad_length + seq)
            else:
                padded_sequences.append(seq + [value] * pad_length)
    return np.array(padded_sequences)

X_padded_pt = pad_sequences_pt(X_pt, maxlen=maxlen_pt + 1, padding='pre')
y_padded_pt = pad_sequences_pt(y_pt, maxlen=maxlen_pt + 1, padding='pre')



X_tensor = torch.tensor(X_padded_pt, dtype=torch.long)
y_one_hot = F.one_hot(torch.tensor(y_padded_pt, dtype=torch.long),
                      num_classes=vocab_size_pt).float()


class PyTorchRNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size, rnn_type="RNN"):
        super(PyTorchRNN, self).__init__()

        self.rnn_type = rnn_type

        self.embedding = nn.Embedding(vocab_size, embedding_dim)

        # CHANGE 1: Flexible model selection
        # Allow switching between RNN, LSTM, GRU

        if rnn_type == "RNN":
            self.rnn = nn.RNN(embedding_dim, hidden_size, batch_first=True)

        elif rnn_type == "LSTM":
            self.rnn = nn.LSTM(embedding_dim, hidden_size, batch_first=True)

        elif rnn_type == "GRU":
            self.rnn = nn.GRU(embedding_dim, hidden_size, batch_first=True)

        else:
            raise ValueError("Choose from RNN / LSTM / GRU")

        self.dense = nn.Linear(hidden_size, vocab_size)

    def forward(self, x):

        embedded = self.embedding(x)

        # CHANGE 2: Handle LSTM output separately
        # LSTM returns (output, (hidden, cell))

        if self.rnn_type == "LSTM":
            output, (hidden, cell) = self.rnn(embedded)
        else:
            output, hidden = self.rnn(embedded)

        output = self.dense(output)

        return output

embedding_dim = 10
hidden_size = 50

# CHANGE 3: Select model here

model_type = "RNN"   # Change to "LSTM" or "GRU"

model_pt = PyTorchRNN(
    vocab_size=vocab_size_pt,
    embedding_dim=embedding_dim,
    hidden_size=hidden_size,
    rnn_type=model_type
)

print("Using Model:", model_type)
print(model_pt)


optimizer_pt = optim.RMSprop(model_pt.parameters())
criterion = nn.CrossEntropyLoss()

y_target_indices = torch.argmax(y_one_hot, dim=-1)

epochs_pt = 200

for epoch in range(epochs_pt):

    optimizer_pt.zero_grad()

    outputs = model_pt(X_tensor)

    outputs_flat = outputs.view(-1, vocab_size_pt)
    targets_flat = y_target_indices.view(-1)

    loss = criterion(outputs_flat, targets_flat)

    loss.backward()
    optimizer_pt.step()

    if (epoch + 1) % 50 == 0:
        print(f'Epoch [{epoch+1}/{epochs_pt}], Loss: {loss.item():.4f}')

print("\n--- Training Complete ---")



def prob_of_input_sentence_pt(model, tokenizer, sentence, vocab_size):
    model.eval()
    with torch.no_grad():

        encoded_list = tokenizer.texts_to_sequences([sentence])[0]

        input_seq = [0] + encoded_list

        padded_input = pad_sequences_pt([input_seq],
                                        maxlen=maxlen_pt + 1,
                                        padding='pre')[0]

        encoded_tensor = torch.tensor(padded_input,
                                      dtype=torch.long).unsqueeze(0)

        logits = model(encoded_tensor)
        prob = F.softmax(logits, dim=-1)

        probability = 1.0

        for i in range(prob.shape[1] - 1):

            target_word_index = int(encoded_tensor[0, i+1].item())

            if target_word_index == 0:
                word_prob = 1.0
            else:
                word_prob = prob[0, i, target_word_index].item()

            probability *= word_prob

        print(f"Sentence: \"{sentence}\" → Probability: {probability}")


print("\n--- Sentence Probabilities ---")

prob_of_input_sentence_pt(model_pt, tokenizer_pt, "Jack and Jill Went", vocab_size_pt)
prob_of_input_sentence_pt(model_pt, tokenizer_pt, "went up the hill", vocab_size_pt)
prob_of_input_sentence_pt(model_pt, tokenizer_pt, "jack fell down and broke", vocab_size_pt)

<>:14: SyntaxWarning: invalid escape sequence '\]'
<>:14: SyntaxWarning: invalid escape sequence '\]'
/tmp/ipykernel_10858/1006749377.py:14: SyntaxWarning: invalid escape sequence '\]'
  def __init__(self, filters='!"#$%&()*+,-/:;<=>?@[\]^_`{|}~'):


Using Model: RNN
PyTorchRNN(
  (embedding): Embedding(24, 10)
  (rnn): RNN(10, 50, batch_first=True)
  (dense): Linear(in_features=50, out_features=24, bias=True)
)
Epoch [50/200], Loss: 0.1861
Epoch [100/200], Loss: 0.1777
Epoch [150/200], Loss: 0.1802
Epoch [200/200], Loss: 0.1771

--- Training Complete ---

--- Sentence Probabilities ---
Sentence: "Jack and Jill Went" → Probability: 0.09629657134570604
Sentence: "went up the hill" → Probability: 1.3889737672936778e-05
Sentence: "jack fell down and broke" → Probability: 2.946342677556137e-05


In [ ]:
# Sentiment Analysis Using RNN, LSTM and GRU

In [8]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from collections import Counter

# Select Model Type
MODEL_TYPE = "rnn"   # options: "rnn", "lstm", "gru"

# Load IMDB Dataset
dataset = load_dataset("imdb")

train_texts = dataset['train']['text']
train_labels = dataset['train']['label']

test_texts = dataset['test']['text']
test_labels = dataset['test']['label']

# Build Vocabulary
def build_vocab(texts, max_size=10000):

    words = []

    for text in texts:
        words.extend(text.lower().split())

    freq = Counter(words)

    vocab = {
        word: i + 1
        for i, (word, _) in enumerate(freq.most_common(max_size))
    }

    return vocab


vocab = build_vocab(train_texts)

# Encode Text
def encode(text, vocab, max_len=100):

    tokens = text.lower().split()

    idxs = [vocab.get(token, 0) for token in tokens]

    if len(idxs) < max_len:
        idxs += [0] * (max_len - len(idxs))
    else:
        idxs = idxs[:max_len]

    return idxs


# Dataset Class
class TextDataset(Dataset):

    def __init__(self, texts, labels, vocab):
        self.texts = texts
        self.labels = labels
        self.vocab = vocab

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):

        x = torch.tensor(
            encode(self.texts[idx], self.vocab),
            dtype=torch.long
        )

        y = torch.tensor(self.labels[idx], dtype=torch.long)

        return x, y


# Create Dataset + Loader
train_dataset = TextDataset(train_texts, train_labels, vocab)
test_dataset = TextDataset(test_texts, test_labels, vocab)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64)


# Model Class (RNN/LSTM/GRU)
class TextClassifier(nn.Module):

    def __init__(self, vocab_size, model_type="rnn",
                 embed_dim=50, hidden_dim=64, output_dim=2):

        super().__init__()

        self.model_type = model_type

        self.embedding = nn.Embedding(
            vocab_size + 1,
            embed_dim,
            padding_idx=0
        )

        if model_type == "rnn":
            self.recurrent = nn.RNN(
                embed_dim,
                hidden_dim,
                batch_first=True
            )

        elif model_type == "lstm":
            self.recurrent = nn.LSTM(
                embed_dim,
                hidden_dim,
                batch_first=True
            )

        elif model_type == "gru":
            self.recurrent = nn.GRU(
                embed_dim,
                hidden_dim,
                batch_first=True
            )

        self.dropout = nn.Dropout(0.5)

        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):

        x = self.embedding(x)

        if self.model_type == "lstm":

            out, (hidden, cell) = self.recurrent(x)
            hidden = hidden[-1]

        else:

            out, hidden = self.recurrent(x)
            hidden = hidden[-1]

        hidden = self.dropout(hidden)

        out = self.fc(hidden)

        return out


# Model Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = TextClassifier(
    vocab_size=len(vocab),
    model_type=MODEL_TYPE
)

model.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)


# Training Function
def train_epoch(model, loader, optimizer, criterion, device):

    model.train()

    total_loss = 0

    for x_batch, y_batch in loader:

        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        outputs = model(x_batch)

        loss = criterion(outputs, y_batch)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)


# Evaluation Function
def evaluate(model, loader, criterion, device):

    model.eval()

    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():

        for x_batch, y_batch in loader:

            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)

            outputs = model(x_batch)

            loss = criterion(outputs, y_batch)

            total_loss += loss.item()

            preds = outputs.argmax(dim=1)

            correct += (preds == y_batch).sum().item()

            total += y_batch.size(0)

    return total_loss / len(loader), correct / total


# Training Loop
epochs = 10

for epoch in range(epochs):

    train_loss = train_epoch(
        model,
        train_loader,
        optimizer,
        criterion,
        device
    )

    val_loss, val_acc = evaluate(
        model,
        test_loader,
        criterion,
        device
    )

    print(
        f"Epoch {epoch+1}: "
        f"Train loss={train_loss:.4f}, "
        f"Val loss={val_loss:.4f}, "
        f"Val acc={val_acc:.4f}"
    )

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Epoch 1: Train loss=0.7004, Val loss=0.6938, Val acc=0.5093
Epoch 2: Train loss=0.6906, Val loss=0.6831, Val acc=0.5632
Epoch 3: Train loss=0.6858, Val loss=0.6912, Val acc=0.5329
Epoch 4: Train loss=0.6819, Val loss=0.6928, Val acc=0.5442
Epoch 5: Train loss=0.6705, Val loss=0.6923, Val acc=0.5479
Epoch 6: Train loss=0.6596, Val loss=0.6940, Val acc=0.5478
Epoch 7: Train loss=0.6643, Val loss=0.6981, Val acc=0.5258
Epoch 8: Train loss=0.6720, Val loss=0.7015, Val acc=0.5296
Epoch 9: Train loss=0.6476, Val loss=0.6870, Val acc=0.5919
Epoch 10: Train loss=0.6260, Val loss=0.6632, Val acc=0.6384
